# Regression by Minimizing Absolute Deviation — Experimental Workflow

This notebook contains the empirical OLS/LAD comparison, error-distribution experiment, runtime benchmark, implementation validation, and retained HBK visualization.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Image

from absolute_deviation.data import DATASETS, load_dataset
from absolute_deviation.experiments import (
    FIGURE_DIR,
    RESULT_DIR,
    run_original_data,
    run_error_distribution_experiment,
    run_runtime_benchmark,
    validate_lad_solver,
)
from absolute_deviation.plotting import generate_all_figures


## 1. Data preparation


In [ ]:
dataset_rows = []
for name in DATASETS:
    X, y, predictors = load_dataset(name)
    dataset_rows.append({
        'dataset': name,
        'observations': len(y),
        'predictors': X.shape[1],
        'predictor_names': ', '.join(predictors),
    })
display(pd.DataFrame(dataset_rows))


## 2. Fit OLS and LAD on the empirical datasets


In [ ]:
metrics_df, coefficients_df = run_original_data()
display(metrics_df[['dataset', 'model', 'SSE', 'SAE']])


## 3. HBK multivariate residual visualization


In [ ]:
generate_all_figures()
display(Image(filename=str(FIGURE_DIR / 'hbk_multivariate_inlier_outlier.png')))


## 4. Error-distribution experiment


In [ ]:
distribution_df = run_error_distribution_experiment()
distribution_summary = (
    distribution_df.groupby(['distribution', 'model'])[['SSE', 'SAE', 'coefficient_error_l2']]
    .median()
    .reset_index()
)
display(distribution_summary)


## 5. Runtime benchmark


In [ ]:
runtime_df = run_runtime_benchmark()
runtime_summary = (
    runtime_df.groupby(['n', 'p', 'model'])['runtime_seconds']
    .median()
    .reset_index()
)
display(runtime_summary)


## 6. Implementation validation


In [ ]:
validation_df = validate_lad_solver()
display(validation_df)


## 7. Generated outputs


In [ ]:
result_files = sorted(path.name for path in RESULT_DIR.glob('*'))
figure_files = sorted(path.name for path in FIGURE_DIR.glob('*'))
display(pd.DataFrame({'result_files': pd.Series(result_files)}))
display(pd.DataFrame({'figure_files': pd.Series(figure_files)}))
